# Minimum-weight Hamiltonian E-to-E path

This notebook is a companion to `ee_backbone_method_comparison.ipynb`. It uses the same excitatory-only matrix, `matrices/mij_EE_matrix.csv`, and the same positive off-diagonal transition graph, but it flips the Hamiltonian objective: instead of finding the heaviest complete path through all 32 excitatory nodes, it finds a complete path with the smallest possible summed transition weight.

That makes the result a contrast path rather than a backbone. It answers: **if every excitatory node must be visited exactly once, what is the weakest directed route supported by the positive E-to-E graph?**

The notebook solves two versions:

1. **Free-start minimum Hamiltonian path:** the solver chooses both the start and terminus.
2. **Canonical EC-start minimum Hamiltonian path:** the path is forced to start at `EC LI II Pyramidal Fan`, matching the canonical EC start used for side-by-side comparisons.


In [1]:
from pathlib import Path

import pandas as pd

from ee_backbone_analysis import (
    EPS,
    beam_path_plus_local,
    prepare_excitatory_matrix_data,
    total_score,
)

INPUT_CSV = Path("matrices/mij_EE_matrix.csv")
OUTDIR = Path("ee_backbone_min_weight_hamiltonian_outputs")
CANONICAL_EC_START = "EC LI II Pyramidal Fan"
TIME_LIMIT_SECONDS = 120
MIN_BEAM_WIDTH = 10_000
MIN_LOCAL_PASSES = 25


## Load and validate the E-only graph

The preprocessing mirrors the existing method-comparison notebook: all row and column labels in `mij_EE_matrix.csv` are treated as excitatory nodes, negative weights are excluded from the candidate transition graph, and diagonal self-connections are ignored. The Hamiltonian constraint then forces the final path to visit all 32 excitatory nodes exactly once.


In [2]:
data = prepare_excitatory_matrix_data(INPUT_CSV, eps=EPS)
OUTDIR.mkdir(exist_ok=True)

n_nodes = len(data.excitatory_nodes)
n_edges = int((data.transition_weights > EPS).sum())

print(f"E-to-E matrix: {data.full_matrix.shape[0]} x {data.full_matrix.shape[1]}")
print(f"Excitatory nodes retained: {n_nodes}")
print(f"Positive off-diagonal candidate transitions: {n_edges}")
print(f"Canonical EC start present: {CANONICAL_EC_START in data.excitatory_nodes}")


E-to-E matrix: 32 x 32
Excitatory nodes retained: 32
Positive off-diagonal candidate transitions: 355
Canonical EC start present: True


## MILP formulation

The optimization is a minimum-weight asymmetric Hamiltonian path. The graph is directed, so `i -> j` and `j -> i` are different candidate transitions with different weights. Binary edge variables decide which transitions are selected. The model selects exactly `n - 1` directed edges, permits at most one incoming and at most one outgoing edge per node, and uses MTZ-style order constraints to eliminate subtours. With all 32 nodes included, those constraints yield one simple path that visits every excitatory node exactly once.

For the free-start solve, the model lets the solver choose the unique node with no incoming edge. For the canonical EC-start solve, the start is fixed by requiring no incoming edge to `EC LI II Pyramidal Fan`, one outgoing edge from it, and order position zero.


In [3]:
def _solver_status(problem):
    import pulp

    solution_status = getattr(problem, "sol_status", None)
    if solution_status is None:
        return pulp.LpStatus.get(problem.status, str(problem.status)), False
    label = pulp.LpSolution.get(solution_status, str(solution_status))
    return label, solution_status == pulp.LpSolutionOptimal


def minimum_weight_asymmetric_hamiltonian_path(
    data,
    seed_name=None,
    time_limit=120,
    msg=False,
):
    try:
        import pulp
    except ImportError as exc:
        raise ImportError(
            "PuLP is required for the minimum-weight Hamiltonian path MILP. "
            "Install it in the notebook kernel with `python -m pip install pulp`."
        ) from exc

    n = len(data.excitatory_nodes)
    if seed_name is not None and seed_name not in data.excitatory_nodes:
        raise ValueError(f"Seed {seed_name!r} is not in the E-only node list.")
    seed = None if seed_name is None else data.excitatory_nodes.index(seed_name)

    edges = [
        (i, j, float(data.transition_weights[i, j]))
        for i in range(n)
        for j in range(n)
        if i != j and data.transition_weights[i, j] > EPS
    ]
    incoming = {i: [] for i in range(n)}
    outgoing = {i: [] for i in range(n)}
    for i, j, _ in edges:
        outgoing[i].append((i, j))
        incoming[j].append((i, j))

    problem = pulp.LpProblem("min_weight_asymmetric_hamiltonian_path", pulp.LpMinimize)
    x = {
        (i, j): pulp.LpVariable(f"x_{i}_{j}", lowBound=0, upBound=1, cat="Binary")
        for i, j, _ in edges
    }
    order = {
        i: pulp.LpVariable(f"u_{i}", lowBound=0, upBound=n - 1, cat="Continuous")
        for i in range(n)
    }

    # Self-weights are constant for a Hamiltonian path and are zero under the
    # current no-diagonal scoring, but including them keeps objective and score aligned
    # if this notebook is later adapted to a self-connection sensitivity run.
    problem += pulp.lpSum(w * x[(i, j)] for i, j, w in edges) + float(data.self_weights.sum())
    problem += pulp.lpSum(x[(i, j)] for i, j, _ in edges) == n - 1

    for i in range(n):
        problem += pulp.lpSum(x[e] for e in incoming[i]) <= 1
        problem += pulp.lpSum(x[e] for e in outgoing[i]) <= 1

    if seed is not None:
        problem += pulp.lpSum(x[e] for e in incoming[seed]) == 0
        if n > 1:
            problem += pulp.lpSum(x[e] for e in outgoing[seed]) == 1
        problem += order[seed] == 0
        for i in range(n):
            if i != seed:
                problem += pulp.lpSum(x[e] for e in incoming[i]) == 1
                problem += order[i] >= 1

    big_m = n
    for i, j, _ in edges:
        problem += order[j] >= order[i] + 1 - big_m * (1 - x[(i, j)])

    solver = pulp.PULP_CBC_CMD(msg=msg, timeLimit=time_limit)
    problem.solve(solver)

    status, proven_optimal = _solver_status(problem)
    selected_edges = {
        i: j
        for (i, j), var in x.items()
        if var.value() is not None and var.value() > 0.5
    }
    selected_receivers = set(selected_edges.values())
    starts = [i for i in range(n) if i not in selected_receivers]
    if seed is not None:
        starts = [seed]
    if len(starts) != 1:
        raise ValueError(f"Expected one Hamiltonian path start, found {len(starts)}: {starts}")

    path = [starts[0]]
    visited = {starts[0]}
    while path[-1] in selected_edges:
        nxt = selected_edges[path[-1]]
        if nxt in visited:
            raise ValueError("MILP returned a cycle despite subtour constraints.")
        path.append(nxt)
        visited.add(nxt)

    if len(path) != n:
        raise ValueError(f"Expected a {n}-node Hamiltonian path, reconstructed {len(path)} nodes.")

    objective = float(pulp.value(problem.objective) or 0.0)
    score = total_score(path, data.transition_weights, data.self_weights)
    return path, score, status, objective, proven_optimal


def path_record(label, path, score, status, objective, proven_optimal):
    names = [data.excitatory_nodes[i] for i in path]
    return {
        "solve": label,
        "start": names[0],
        "terminus": names[-1],
        "length_edges": len(path) - 1,
        "n_nodes_visited": len(path),
        "summed_weight": score,
        "objective": objective,
        "solver_status": status,
        "proven_optimal": proven_optimal,
        "path": " -> ".join(names),
    }


def edge_records(label, path):
    records = []
    for step, (u, v) in enumerate(zip(path[:-1], path[1:]), start=1):
        records.append(
            {
                "solve": label,
                "step": step,
                "sender": data.excitatory_nodes[u],
                "receiver": data.excitatory_nodes[v],
                "weight": float(data.transition_weights[u, v]),
            }
        )
    return records


## Beam plus local heuristic for the weakest complete path

The exact MILP below is the authoritative formulation when it proves optimality. This section adds a faster heuristic analogue: a beam search keeps the lowest-weight partial Hamiltonian paths, then a local relocation pass moves already-used nodes to different positions whenever the move lowers total weight and preserves a valid directed path.

This is the minimum-weight counterpart to `beam_path_plus_local` in the main E-to-E comparison notebook. Because every node must be used, the local step cannot insert unused nodes; instead, it relocates nodes within the 32-node ordering. The result is a useful comparator or warm-start-style candidate, not an optimality certificate.


In [4]:
def _path_is_valid(path):
    return all(data.transition_weights[u, v] > EPS for u, v in zip(path[:-1], path[1:]))


def _path_weight(path):
    if not _path_is_valid(path):
        return float("inf")
    return total_score(path, data.transition_weights, data.self_weights)


def _beam_min_hamiltonian_path(data, seed_name=None, beam_width=10_000):
    n = len(data.excitatory_nodes)
    if seed_name is None:
        states = [
            (1 << start, start, float(data.self_weights[start]), [start])
            for start in range(n)
        ]
    else:
        if seed_name not in data.excitatory_nodes:
            raise ValueError(f"Seed {seed_name!r} is not in the E-only node list.")
        start = data.excitatory_nodes.index(seed_name)
        states = [(1 << start, start, float(data.self_weights[start]), [start])]

    for _ in range(n - 1):
        candidates = []
        for mask, last, score, path in states:
            for nxt, weight in data.adjacency[last]:
                if mask & (1 << nxt):
                    continue
                candidates.append(
                    (
                        mask | (1 << nxt),
                        nxt,
                        score + float(weight) + float(data.self_weights[nxt]),
                        path + [nxt],
                    )
                )
        if not candidates:
            raise ValueError("Beam search could not complete a Hamiltonian path with the current graph and beam width.")
        candidates.sort(key=lambda item: (item[2], len(item[3])))
        states = candidates[:beam_width]

    complete = [state for state in states if len(state[3]) == n]
    if not complete:
        raise ValueError("Beam search ended without a complete Hamiltonian path.")
    _, _, score, path = min(complete, key=lambda item: item[2])
    return path, float(score)


def _local_relocate_min_path(path, fixed_start=False, max_passes=25):
    path = list(path)
    best_score = _path_weight(path)
    min_insert_pos = 1 if fixed_start else 0

    for _ in range(max_passes):
        best_candidate = None
        best_candidate_score = best_score

        for remove_pos in range(len(path)):
            if fixed_start and remove_pos == 0:
                continue
            node = path[remove_pos]
            remainder = path[:remove_pos] + path[remove_pos + 1 :]
            for insert_pos in range(min_insert_pos, len(remainder) + 1):
                if insert_pos == remove_pos or insert_pos == remove_pos + 1:
                    continue
                candidate = remainder[:insert_pos] + [node] + remainder[insert_pos:]
                candidate_score = _path_weight(candidate)
                if candidate_score + EPS < best_candidate_score:
                    best_candidate = candidate
                    best_candidate_score = candidate_score

        if best_candidate is None:
            break
        path = best_candidate
        best_score = best_candidate_score

    return path, best_score


def min_weight_hamiltonian_beam_plus_local(
    data,
    seed_name=None,
    beam_width=10_000,
    max_local_passes=25,
):
    beam_path, beam_score = _beam_min_hamiltonian_path(
        data,
        seed_name=seed_name,
        beam_width=beam_width,
    )
    improved_path, improved_score = _local_relocate_min_path(
        beam_path,
        fixed_start=seed_name is not None,
        max_passes=max_local_passes,
    )
    return improved_path, improved_score, beam_score


## Solve the beam-plus-local heuristic paths

These heuristic rows are available even before the MILP finishes. They visit all 32 excitatory nodes exactly once, but `proven_optimal` is `False` because beam pruning and local relocation do not provide a proof of global minimality.


In [5]:
beam_local_solves = [
    ("free_start_beam_plus_local", None),
    ("canonical_ec_start_beam_plus_local", CANONICAL_EC_START),
]

beam_local_path_rows = []
beam_local_edge_rows = []

for label, seed_name in beam_local_solves:
    path, score, beam_score = min_weight_hamiltonian_beam_plus_local(
        data,
        seed_name=seed_name,
        beam_width=MIN_BEAM_WIDTH,
        max_local_passes=MIN_LOCAL_PASSES,
    )
    record = path_record(label, path, score, "heuristic", score, False)
    record["beam_initial_score"] = beam_score
    record["local_improvement"] = beam_score - score
    beam_local_path_rows.append(record)
    beam_local_edge_rows.extend(edge_records(label, path))

beam_local_minimum_paths = pd.DataFrame(beam_local_path_rows)
beam_local_minimum_edges = pd.DataFrame(beam_local_edge_rows)

beam_local_minimum_paths.to_csv(OUTDIR / "beam_local_minimum_weight_hamiltonian_paths.csv", index=False)
beam_local_minimum_edges.to_csv(OUTDIR / "beam_local_minimum_weight_hamiltonian_edges.csv", index=False)

beam_local_minimum_paths[
    [
        "solve",
        "start",
        "terminus",
        "length_edges",
        "n_nodes_visited",
        "summed_weight",
        "beam_initial_score",
        "local_improvement",
        "proven_optimal",
    ]
]


,solve,start,terminus,length_edges,n_nodes_visited,summed_weight,beam_initial_score,local_improvement,proven_optimal
0,free_start_beam_plus_local,MEC LIII Bipolar Complex Pyramidal,CA3 Pyramidal,31,32,18008.864423,19275.323767,1266.459344,False
1,canonical_ec_start_beam_plus_local,EC LI II Pyramidal Fan,CA3 Pyramidal,31,32,17625.553009,17741.914671,116.361662,False


## Solve the weakest complete routes

The free-start route is the literal minimum over all directed Hamiltonian paths available in the positive E-to-E graph. The canonical EC-start route answers the same minimization question, but under a fixed biological starting point. A solver status marked as a proven optimum is required before treating either result as exact.


In [6]:
solves = [
    ("free_start_minimum", None),
    ("canonical_ec_start_minimum", CANONICAL_EC_START),
]

path_rows = []
edge_rows = []

try:
    for label, seed_name in solves:
        path, score, status, objective, proven_optimal = minimum_weight_asymmetric_hamiltonian_path(
            data,
            seed_name=seed_name,
            time_limit=TIME_LIMIT_SECONDS,
            msg=False,
        )
        path_rows.append(path_record(label, path, score, status, objective, proven_optimal))
        edge_rows.extend(edge_records(label, path))
except ImportError as exc:
    print(exc)
    print("Skipping exact MILP rows; beam-plus-local heuristic rows remain available below.")

minimum_path_csv = OUTDIR / "minimum_weight_hamiltonian_paths.csv"
minimum_edge_csv = OUTDIR / "minimum_weight_hamiltonian_edges.csv"

minimum_paths = pd.DataFrame(path_rows)
minimum_edges = pd.DataFrame(edge_rows)

if not minimum_paths.empty:
    minimum_paths.to_csv(minimum_path_csv, index=False)
elif minimum_path_csv.exists():
    print(f"Loading existing exact MILP path rows from {minimum_path_csv}.")
    minimum_paths = pd.read_csv(minimum_path_csv)

if not minimum_edges.empty:
    minimum_edges.to_csv(minimum_edge_csv, index=False)
elif minimum_edge_csv.exists():
    print(f"Loading existing exact MILP edge rows from {minimum_edge_csv}.")
    minimum_edges = pd.read_csv(minimum_edge_csv)

minimum_paths[
    [
        "solve",
        "start",
        "terminus",
        "length_edges",
        "n_nodes_visited",
        "summed_weight",
        "solver_status",
        "proven_optimal",
    ]
] if not minimum_paths.empty else minimum_paths


PuLP is required for the minimum-weight Hamiltonian path MILP. Install it in the notebook kernel with `python -m pip install pulp`.
Skipping exact MILP rows; beam-plus-local heuristic rows remain available below.
Loading existing exact MILP path rows from ee_backbone_min_weight_hamiltonian_outputs\minimum_weight_hamiltonian_paths.csv.
Loading existing exact MILP edge rows from ee_backbone_min_weight_hamiltonian_outputs\minimum_weight_hamiltonian_edges.csv.


,solve,start,terminus,length_edges,n_nodes_visited,summed_weight,solver_status,proven_optimal
0,free_start_minimum,CA2 Pyramidal,CA3 Pyramidal,31,32,10277.163873,Optimal Solution Found,True
1,canonical_ec_start_minimum,EC LI II Pyramidal Fan,CA3 Pyramidal,31,32,12715.218809,Optimal Solution Found,True


## Inspect the selected paths

These transposed tables show the full selected route for each solve. Because the objective minimizes weight while still forcing all 32 nodes to be visited, this output should be interpreted as the weakest complete contrast path, not as a high-weight excitatory backbone.


In [7]:
for _, row in minimum_paths.iterrows():
    print(row["solve"])
    display(
        row[
            [
                "start",
                "terminus",
                "length_edges",
                "n_nodes_visited",
                "summed_weight",
                "solver_status",
                "proven_optimal",
                "path",
            ]
        ].to_frame(name="value")
    )


free_start_minimum
canonical_ec_start_minimum


,value
start,CA2 Pyramidal
terminus,CA3 Pyramidal
length_edges,31
n_nodes_visited,32
summed_weight,10277.163873
solver_status,Optimal Solution Found
proven_optimal,True
path,CA2 Pyramidal -> CA3 Granule -> CA3c Pyramidal...


,value
start,EC LI II Pyramidal Fan
terminus,CA3 Pyramidal
length_edges,31
n_nodes_visited,32
summed_weight,12715.218809
solver_status,Optimal Solution Found
proven_optimal,True
path,EC LI II Pyramidal Fan -> EC LV Deep Pyramidal...


## Edge-level output

The edge table below is useful for checking which low-weight transitions make the complete route possible. It is also the easiest place to review whether any selected transition is biologically implausible or likely to be an artifact of the matrix construction.


In [8]:
minimum_edges.head(40)


,solve,step,sender,receiver,weight
0,free_start_minimum,1,CA2 Pyramidal,CA3 Granule,592.986335
1,free_start_minimum,2,CA3 Granule,CA3c Pyramidal,2.280811
2,free_start_minimum,3,CA3c Pyramidal,SUB CA1 Projecting Pyramidal,1119.440223
3,free_start_minimum,4,SUB CA1 Projecting Pyramidal,CA1 Radiatum Giant,8.287726
4,free_start_minimum,5,CA1 Radiatum Giant,CA1 Pyramidal,52.526068
5,free_start_minimum,6,CA1 Pyramidal,LEC LIII Multipolar Principal,3004.906539
6,free_start_minimum,7,LEC LIII Multipolar Principal,LEC LVI Multipolar Pyramidal,32.632235
7,free_start_minimum,8,LEC LVI Multipolar Pyramidal,EC LIII Stellate,43.439814
8,free_start_minimum,9,EC LIII Stellate,MEC LIII Complex Pyramidal,165.812229
9,free_start_minimum,10,MEC LIII Complex Pyramidal,EC LIII V Bipolar Pyramidal,70.676774


## Write outputs and final printout tables

This section follows the output pattern from `ee_backbone_method_comparison.ipynb`: it gathers the exact and heuristic minimum-weight Hamiltonian rows, writes compact comparison artifacts, and then displays the final tables a reader is most likely to inspect after rerunning the notebook.

In [9]:
MINIMUM_OUTPUT_FILES = [
    "minimum_weight_hamiltonian_paths.csv",
    "minimum_weight_hamiltonian_edges.csv",
    "beam_local_minimum_weight_hamiltonian_paths.csv",
    "beam_local_minimum_weight_hamiltonian_edges.csv",
]

METHOD_TABLE_ORDER = [
    "greedy_tree_path",
    "maximum_spanning_tree",
    "branch_and_bound",
    "dynamic_programming",
    "beam_path_plus_local",
    "milp_subtour_elimination",
    "maximum_weight_asymmetric_hamiltonian_path",
    "feedback_arc_ordering",
    "canonical_ec_start_minimum",
    "canonical_ec_start_beam_plus_local",
    "free_start_minimum",
    "free_start_beam_plus_local",
]

METHOD_TABLE_LABELS = {
    "greedy_tree_path": "Greedy tree/path",
    "maximum_spanning_tree": "Maximum spanning tree",
    "branch_and_bound": "Branch-and-bound",
    "dynamic_programming": "Dynamic programming / beam DP",
    "beam_path_plus_local": "Beam path plus local",
    "milp_subtour_elimination": "MILP subtour elimination",
    "maximum_weight_asymmetric_hamiltonian_path": "Maximum-weight Hamiltonian path",
    "feedback_arc_ordering": "Feedback-arc ordering path",
    "canonical_ec_start_minimum": "Minimum-weight Hamiltonian path",
    "canonical_ec_start_beam_plus_local": "Min Hamiltonian beam plus local",
    "free_start_minimum": "Minimum-weight Hamiltonian path, free start",
    "free_start_beam_plus_local": "Min Hamiltonian beam plus local, free start",
}

PATH_TABLE_ROWS = [
    "source",
    "start",
    "terminus",
    "length_edges",
    "n_nodes_visited",
    "summed_weight",
    "proven_optimal",
    "path",
]


def _read_csv_if_exists(csv_path: Path) -> pd.DataFrame:
    return pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()


def _minimum_hamiltonian_rows() -> pd.DataFrame:
    row_sets = []

    if "minimum_paths" in globals() and not minimum_paths.empty:
        row_sets.append(minimum_paths.copy())
    else:
        rows = _read_csv_if_exists(OUTDIR / "minimum_weight_hamiltonian_paths.csv")
        if not rows.empty:
            row_sets.append(rows)

    if "beam_local_minimum_paths" in globals() and not beam_local_minimum_paths.empty:
        row_sets.append(beam_local_minimum_paths.copy())
    else:
        rows = _read_csv_if_exists(OUTDIR / "beam_local_minimum_weight_hamiltonian_paths.csv")
        if not rows.empty:
            row_sets.append(rows)

    if not row_sets:
        return pd.DataFrame()

    rows = pd.concat(row_sets, ignore_index=True, sort=False)
    rows["method_id"] = rows["solve"]
    rows["method_label"] = rows["method_id"].map(METHOD_TABLE_LABELS).fillna(rows["method_id"])
    rows["source"] = "minimum-weight notebook"
    rows["objective_direction"] = "minimize summed_weight"
    rows["start_constraint"] = rows["method_id"].map(
        lambda value: "canonical EC start" if str(value).startswith("canonical_ec_start") else "free start"
    )
    return rows


def _main_method_path_rows(seed_name: str) -> pd.DataFrame:
    comparison_path = Path("ee_backbone_comparison_outputs/method_comparison_long.csv")
    rows = _read_csv_if_exists(comparison_path)
    if rows.empty:
        return rows

    rows = rows[rows["seed"].eq(seed_name)].copy()
    if rows.empty:
        return rows

    rows["method_id"] = rows["method"]
    rows["method_label"] = rows["method_id"].map(METHOD_TABLE_LABELS).fillna(rows["method_id"])
    rows["source"] = "main method comparison"
    rows["start"] = rows["seed"]
    rows["proven_optimal"] = ""
    rows["objective_direction"] = "maximize summed_weight"

    if "milp_proven_optimal" in rows.columns:
        mask = rows["method_id"].eq("milp_subtour_elimination")
        rows.loc[mask, "proven_optimal"] = rows.loc[mask, "milp_proven_optimal"].astype("string")
    if "hamiltonian_proven_optimal" in rows.columns:
        mask = rows["method_id"].eq("maximum_weight_asymmetric_hamiltonian_path")
        rows.loc[mask, "proven_optimal"] = rows.loc[mask, "hamiltonian_proven_optimal"].astype("string")
    if "ordering_exact_within_ordering" in rows.columns:
        mask = rows["method_id"].eq("feedback_arc_ordering")
        rows.loc[mask, "proven_optimal"] = rows.loc[mask, "ordering_exact_within_ordering"].astype("string")

    return rows


def _method_summary(rows: pd.DataFrame) -> pd.DataFrame:
    if rows.empty:
        return rows

    summary = (
        rows.groupby(["method_id", "method_label", "objective_direction", "start_constraint"], dropna=False)
        .agg(
            n_paths=("method_id", "size"),
            min_summed_weight=("summed_weight", "min"),
            mean_summed_weight=("summed_weight", "mean"),
            max_summed_weight=("summed_weight", "max"),
            min_length_edges=("length_edges", "min"),
            max_length_edges=("length_edges", "max"),
            n_proven_optimal=("proven_optimal", lambda values: sum(str(value).lower() == "true" for value in values)),
        )
        .reset_index()
    )
    summary["method_order"] = summary["method_id"].map(
        {method: idx for idx, method in enumerate(METHOD_TABLE_ORDER)}
    ).fillna(len(METHOD_TABLE_ORDER))
    return summary.sort_values(["method_order", "start_constraint"]).drop(columns="method_order")


def _method_column_path_table(rows: pd.DataFrame) -> pd.DataFrame:
    if rows.empty:
        return pd.DataFrame()

    rows = rows.copy()
    rows["method_order"] = rows["method_id"].map(
        {method: idx for idx, method in enumerate(METHOD_TABLE_ORDER)}
    ).fillna(len(METHOD_TABLE_ORDER))
    rows = rows.sort_values("method_order")

    table = rows.set_index("method_label")[PATH_TABLE_ROWS].T
    if "summed_weight" in table.index:
        table.loc["summed_weight"] = table.loc["summed_weight"].map(
            lambda value: f"{float(value):,.2f}" if pd.notna(value) and value != "" else value
        )
    return table


def _edge_walk_table(solve_name: str) -> pd.DataFrame:
    if "minimum_edges" in globals() and not minimum_edges.empty:
        edge_table = minimum_edges.copy()
    else:
        edge_table = _read_csv_if_exists(OUTDIR / "minimum_weight_hamiltonian_edges.csv")
    if edge_table.empty:
        return edge_table

    walk = edge_table[edge_table["solve"].eq(solve_name)].copy()
    if walk.empty:
        return walk
    walk["cumulative_weight"] = walk["weight"].cumsum()
    return walk[["step", "sender", "receiver", "weight", "cumulative_weight"]]


def _saved_file_inventory() -> pd.DataFrame:
    records = []
    for name in MINIMUM_OUTPUT_FILES + [
        "minimum_weight_path_comparison.csv",
        "minimum_weight_method_summary.csv",
        "canonical_ec_method_path_table.csv",
        "free_start_minimum_path_table.csv",
        "canonical_ec_minimum_edge_walk.csv",
    ]:
        csv_path = OUTDIR / name
        if csv_path.exists():
            records.append(
                {
                    "file": str(csv_path),
                    "rows": len(pd.read_csv(csv_path)),
                    "modified": pd.Timestamp.fromtimestamp(csv_path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
                }
            )
    return pd.DataFrame(records)


minimum_hamiltonian_path_rows = _minimum_hamiltonian_rows()
main_method_path_rows = _main_method_path_rows(CANONICAL_EC_START)
canonical_minimum_rows = minimum_hamiltonian_path_rows[
    minimum_hamiltonian_path_rows["method_id"].astype(str).str.startswith("canonical_ec_start")
].copy()
free_start_minimum_rows = minimum_hamiltonian_path_rows[
    minimum_hamiltonian_path_rows["method_id"].astype(str).str.startswith("free_start")
].copy()

minimum_method_summary = _method_summary(minimum_hamiltonian_path_rows)
canonical_ec_method_path_table = _method_column_path_table(
    pd.concat([main_method_path_rows, canonical_minimum_rows], ignore_index=True, sort=False)
)
free_start_minimum_path_table = _method_column_path_table(free_start_minimum_rows)
canonical_ec_minimum_edge_walk = _edge_walk_table("canonical_ec_start_minimum")

if not minimum_hamiltonian_path_rows.empty:
    minimum_hamiltonian_path_rows.to_csv(OUTDIR / "minimum_weight_path_comparison.csv", index=False)
if not minimum_method_summary.empty:
    minimum_method_summary.to_csv(OUTDIR / "minimum_weight_method_summary.csv", index=False)
if not canonical_ec_method_path_table.empty:
    canonical_ec_method_path_table.to_csv(OUTDIR / "canonical_ec_method_path_table.csv")
if not free_start_minimum_path_table.empty:
    free_start_minimum_path_table.to_csv(OUTDIR / "free_start_minimum_path_table.csv")
if not canonical_ec_minimum_edge_walk.empty:
    canonical_ec_minimum_edge_walk.to_csv(OUTDIR / "canonical_ec_minimum_edge_walk.csv", index=False)

output_inventory = _saved_file_inventory()
print("Wrote / found:")
for name in output_inventory["file"].tolist() if not output_inventory.empty else []:
    print(f"  {name}")

output_inventory

Wrote / found:
  ee_backbone_min_weight_hamiltonian_outputs\minimum_weight_hamiltonian_paths.csv
  ee_backbone_min_weight_hamiltonian_outputs\minimum_weight_hamiltonian_edges.csv
  ee_backbone_min_weight_hamiltonian_outputs\beam_local_minimum_weight_hamiltonian_paths.csv
  ee_backbone_min_weight_hamiltonian_outputs\beam_local_minimum_weight_hamiltonian_edges.csv
  ee_backbone_min_weight_hamiltonian_outputs\minimum_weight_path_comparison.csv
  ee_backbone_min_weight_hamiltonian_outputs\minimum_weight_method_summary.csv
  ee_backbone_min_weight_hamiltonian_outputs\canonical_ec_method_path_table.csv
  ee_backbone_min_weight_hamiltonian_outputs\free_start_minimum_path_table.csv
  ee_backbone_min_weight_hamiltonian_outputs\canonical_ec_minimum_edge_walk.csv


,file,rows,modified
0,ee_backbone_min_weight_hamiltonian_outputs\min...,2,2026-09-17 17:25:58
1,ee_backbone_min_weight_hamiltonian_outputs\min...,62,2026-09-17 17:25:58
2,ee_backbone_min_weight_hamiltonian_outputs\bea...,2,2026-09-28 13:56:58
3,ee_backbone_min_weight_hamiltonian_outputs\bea...,62,2026-09-28 13:56:58
4,ee_backbone_min_weight_hamiltonian_outputs\min...,4,2026-09-28 13:56:58
5,ee_backbone_min_weight_hamiltonian_outputs\min...,4,2026-09-28 13:56:58
6,ee_backbone_min_weight_hamiltonian_outputs\can...,8,2026-09-28 13:56:58
7,ee_backbone_min_weight_hamiltonian_outputs\fre...,8,2026-09-28 13:56:58
8,ee_backbone_min_weight_hamiltonian_outputs\can...,31,2026-09-28 13:56:58


## Final summary tables

The first table summarizes the minimum-weight Hamiltonian solves. The second table places the canonical EC-start minimum-weight contrast next to the maximum-weight and ordering methods from `ee_backbone_method_comparison.ipynb`. The third table keeps the free-start minimum variants separate because they answer a different start-condition question.

In [10]:
def _display_path_table(title: str, table: pd.DataFrame) -> None:
    print(title)
    if table.empty:
        print("No rows available. Run the solve cells above, or rerun the main method-comparison notebook to regenerate comparison outputs.")
    else:
        with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
            display(table)


_display_path_table("Minimum-weight Hamiltonian solve summary", minimum_method_summary)
_display_path_table("Canonical EC-start paths: main methods plus minimum-weight contrasts", canonical_ec_method_path_table)
_display_path_table("Free-start minimum-weight contrast paths", free_start_minimum_path_table)

Minimum-weight Hamiltonian solve summary
Canonical EC-start paths: main methods plus minimum-weight contrasts
Free-start minimum-weight contrast paths


,method_id,method_label,objective_direction,start_constraint,n_paths,min_summed_weight,mean_summed_weight,max_summed_weight,min_length_edges,max_length_edges,n_proven_optimal
1,canonical_ec_start_minimum,Minimum-weight Hamiltonian path,minimize summed_weight,canonical EC start,1,12715.218809,12715.218809,12715.218809,31,31,1
0,canonical_ec_start_beam_plus_local,Min Hamiltonian beam plus local,minimize summed_weight,canonical EC start,1,17625.553009,17625.553009,17625.553009,31,31,0
3,free_start_minimum,"Minimum-weight Hamiltonian path, free start",minimize summed_weight,free start,1,10277.163873,10277.163873,10277.163873,31,31,1
2,free_start_beam_plus_local,"Min Hamiltonian beam plus local, free start",minimize summed_weight,free start,1,18008.864423,18008.864423,18008.864423,31,31,0


method_label,Greedy tree/path,Maximum spanning tree,Branch-and-bound,Dynamic programming / beam DP,Beam path plus local,MILP subtour elimination,Maximum-weight Hamiltonian path,Feedback-arc ordering path,Minimum-weight Hamiltonian path,Min Hamiltonian beam plus local
source,main method comparison,main method comparison,main method comparison,main method comparison,main method comparison,main method comparison,main method comparison,main method comparison,minimum-weight notebook,minimum-weight notebook
start,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan,EC LI II Pyramidal Fan
terminus,CA1 Radiatum Giant,MEC LV VI Pyramidal Polymorphic,CA1 Radiatum Giant,CA1 Pyramidal,MEC LV VI Pyramidal Polymorphic,DG Hilar Ectopic Granule,CA1 Radiatum Giant,DG Hilar Ectopic Granule,CA3 Pyramidal,CA3 Pyramidal
length_edges,15,4,30,4,28,28,31,14,31,31
n_nodes_visited,16,5,31,5,29,29,32,15,32,32
summed_weight,"163,171.34","107,708.62","239,308.67","117,456.23","223,975.06","241,901.42","223,058.62","199,781.08","12,715.22","17,625.55"
proven_optimal,,,,,,True,True,True,True,False
path,EC LI II Pyramidal Fan -> EC LI II Multipolar Pyramidal -> MEC LIII Bipolar Complex Pyramidal -> MEC LIII Complex Pyramidal -> MEC LII Stellate -> CA3 Granule -> CA3c Pyramidal -> CA1 Pyramidal -> MEC LV VI Pyramidal Polymorphic -> EC LIV VI Deep Multipolar Principal -> LEC LVI Multipolar Pyramidal -> LEC LIII Multipolar Principal -> LEC LIII Complex Pyramidal -> EC LIII Pyramidal Stellate -> SUB CA1 Projecting Pyramidal -> CA1 Radiatum Giant,EC LI II Pyramidal Fan -> DG Granule -> CA3c Pyramidal -> CA1 Pyramidal -> MEC LV VI Pyramidal Polymorphic,EC LI II Pyramidal Fan -> EC LI II Multipolar Pyramidal -> MEC LIII Bipolar Complex Pyramidal -> MEC LIII Complex Pyramidal -> MEC LII Stellate -> CA3 Granule -> CA3c Pyramidal -> CA1 Pyramidal -> MEC LV VI Pyramidal Polymorphic -> EC LIV VI Deep Multipolar Principal -> LEC LVI Multipolar Pyramidal -> LEC LIII Multipolar Principal -> LEC LIII Complex Pyramidal -> EC LIII Pyramidal Stellate -> EC LII III Pyramidal Tripolar -> EC LIII Stellate -> MEC LII III Pyramidal Multiform -> MEC LV Multipolar Pyramidal -> MEC LV Superficial Pyramidal -> MEC LII Oblique Pyramidal -> EC LIII Pyramidal -> EC LIII V Bipolar Pyramidal -> EC LV Deep Pyramidal -> MEC LV Pyramidal -> DG Granule -> DG Mossy -> DG Semilunar Granule -> DG Hilar Ectopic Granule -> CA3 Pyramidal -> CA2 Pyramidal -> CA1 Radiatum Giant,EC LI II Pyramidal Fan -> DG Granule -> DG Hilar Ectopic Granule -> CA3c Pyramidal -> CA1 Pyramidal,EC LI II Pyramidal Fan -> EC LI II Multipolar Pyramidal -> MEC LIII Bipolar Complex Pyramidal -> MEC LII Oblique Pyramidal -> EC LIII Pyramidal -> EC LIII V Bipolar Pyramidal -> EC LV Deep Pyramidal -> EC LIV VI Deep Multipolar Principal -> LEC LVI Multipolar Pyramidal -> LEC LIII Complex Pyramidal -> LEC LIII Multipolar Principal -> EC LIII Stellate -> EC LIII Pyramidal Stellate -> EC LII III Pyramidal Tripolar -> MEC LII Stellate -> MEC LIII Complex Pyramidal -> MEC LII III Pyramidal Multiform -> MEC LV Multipolar Pyramidal -> MEC LV Superficial Pyramidal -> MEC LV Pyramidal -> DG Granule -> DG Mossy -> DG Semilunar Granule -> DG Hilar Ectopic Granule -> CA3 Pyramidal -> CA2 Pyramidal -> CA3c Pyramidal -> CA1 Pyramidal -> MEC LV VI Pyramidal Polymorphic,EC LI II Pyramidal Fan -> EC LI II Multipolar Pyramidal -> MEC LIII Bipolar Complex Pyramidal -> MEC LIII Complex Pyramidal -> MEC LII Stellate -> CA3 Granule -> CA3 Pyramidal -> CA2 Pyramidal -> CA3c Pyramidal -> CA1 Pyramidal -> MEC LV VI Pyramidal Polymorphic -> EC LV Deep Pyramidal -> EC LIV VI Deep Multipolar Principal -> LEC LVI Multipolar Pyramidal -> EC LIII V Bipolar Pyramidal -> LEC LIII Multipolar Principal -> LEC LIII Complex Pyramidal -> EC LIII Pyramidal Stellate -> EC LII III Pyramidal Tripolar -> EC LIII Stellate

method_label,"Minimum-weight Hamiltonian path, free start","Min Hamiltonian beam plus local, free start"
source,minimum-weight notebook,minimum-weight notebook
start,CA2 Pyramidal,MEC LIII Bipolar Complex Pyramidal
terminus,CA3 Pyramidal,CA3 Pyramidal
length_edges,31,31
n_nodes_visited,32,32
summed_weight,"10,277.16","18,008.86"
proven_optimal,True,False
path,CA2 Pyramidal -> CA3 Granule -> CA3c Pyramidal -> SUB CA1 Projecting Pyramidal -> CA1 Radiatum Giant -> CA1 Pyramidal -> LEC LIII Multipolar Principal -> LEC LVI Multipolar Pyramidal -> EC LIII Stellate -> MEC LIII Complex Pyramidal -> EC LIII V Bipolar Pyramidal -> MEC LII III Pyramidal Multiform -> MEC LII Oblique Pyramidal -> MEC LIII Bipolar Complex Pyramidal -> EC LIII Pyramidal -> MEC LV VI Pyramidal Polymorphic -> EC LIV VI Deep Multipolar Principal -> MEC LV Pyramidal -> EC LII III Pyramidal Tripolar -> EC LI II Pyramidal Fan -> EC LV Deep Pyramidal -> MEC LV Multipolar Pyramidal -> EC LIII Pyramidal Stellate -> LEC LIII Complex Pyramidal -> EC LI II Multipolar Pyramidal -> MEC LV Superficial Pyramidal -> MEC LII Stellate -> DG Hilar Ectopic Granule -> DG Mossy -> DG Semilunar Granule -> DG Granule -> CA3 Pyramidal,MEC LIII Bipolar Complex Pyramidal -> EC LIII Stellate -> LEC LIII Complex Pyramidal -> LEC LVI Multipolar Pyramidal -> EC LIII Pyramidal -> MEC LV VI Pyramidal Polymorphic -> EC LV Deep Pyramidal -> MEC LV Superficial Pyramidal -> MEC LII Oblique Pyramidal -> MEC LII III Pyramidal Multiform -> MEC LII Stellate -> DG Hilar Ectopic Granule -> DG Mossy -> DG Semilunar Granule -> CA3c Pyramidal -> SUB CA1 Projecting Pyramidal -> CA1 Radiatum Giant -> CA1 Pyramidal -> LEC LIII Multipolar Principal -> EC LIV VI Deep Multipolar Principal -> MEC LV Pyramidal -> EC LIII Pyramidal Stellate -> MEC LIII Complex Pyramidal -> EC LI II Multipolar Pyramidal -> EC LI II Pyramidal Fan -> EC LIII V Bipolar Pyramidal -> MEC LV Multipolar Pyramidal -> EC LII III Pyramidal Tripolar -> DG Granule -> CA2 Pyramidal -> CA3 Granule -> CA3 Pyramidal


## Walk the canonical minimum path

This final printout expands the exact canonical EC-start minimum path edge by edge. It is the easiest table to inspect when checking which low-weight transitions make the complete route possible.

In [11]:
_display_path_table("Canonical EC-start minimum-weight edge walk", canonical_ec_minimum_edge_walk)

Canonical EC-start minimum-weight edge walk


,step,sender,receiver,weight,cumulative_weight
31,1,EC LI II Pyramidal Fan,EC LV Deep Pyramidal,426.214928,426.214928
32,2,EC LV Deep Pyramidal,LEC LIII Multipolar Principal,197.502431,623.717359
33,3,LEC LIII Multipolar Principal,LEC LVI Multipolar Pyramidal,32.632235,656.349595
34,4,LEC LVI Multipolar Pyramidal,EC LIII Stellate,43.439814,699.789409
35,5,EC LIII Stellate,MEC LIII Complex Pyramidal,165.812229,865.601638
36,6,MEC LIII Complex Pyramidal,EC LIII V Bipolar Pyramidal,70.676774,936.278411
37,7,EC LIII V Bipolar Pyramidal,MEC LII III Pyramidal Multiform,296.418978,1232.697389
38,8,MEC LII III Pyramidal Multiform,MEC LII Oblique Pyramidal,180.200324,1412.897713
39,9,MEC LII Oblique Pyramidal,MEC LIII Bipolar Complex Pyramidal,57.538994,1470.436708
40,10,MEC LIII Bipolar Complex Pyramidal,MEC LV Multipolar Pyramidal,88.140017,1558.576724


## Takeaways

- This notebook uses the same E-only graph as the main method-comparison notebook, but it intentionally inverts the Hamiltonian objective.
- Every reported Hamiltonian path must contain 31 directed transitions and visit all 32 excitatory nodes exactly once.
- The final tables now mirror the reporting style of `ee_backbone_method_comparison.ipynb`: saved-output inventory, method-level summaries, side-by-side path tables, and an edge-level walk.
- The result is a **minimum-weight complete route**, so it is best used as a contrast or null-style comparator against the maximum-weight Hamiltonian path, not as evidence for a strong backbone.
- If either exact solve is not marked `proven_optimal`, increase `TIME_LIMIT_SECONDS` and rerun before quoting the minimum value.